# 01 — Gold | Team Performance KPIs

**Project:** MLB Performance Intelligence Platform

Answers the team questions in the KPI catalog (TEAM-001 … TEAM-012, H2H-001 … H2H-004). Gold
tables are fully recomputed from Silver on each run (a few thousand rows; always consistent).

| Table | Grain | Business questions |
|---|---|---|
| `gold.team_game_log` | team × game | How did the record, run differential and streak evolve game by game? |
| `gold.team_season_summary` | team × season × season type | Record, home/away and one-run splits, last 10, streaks, **Pythagorean expected wins vs actual** ("luck") |
| `gold.head_to_head` | team × opponent × season × season type | When did two teams meet and who won? |

In [ ]:
import sys, importlib
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
import pipeline as P
for _m in (C, P):
    importlib.reload(_m)

RUN_ID = P.new_run_id()
T = lambda s, n: C.table(s, n)

## team_game_log
Running record, rolling last-10 and the current streak (gaps-and-islands: consecutive games
with the same result share `row_number() - row_number() by result`).

In [ ]:
team_game_log = spark.sql(f"""
WITH g AS (
  SELECT tg.*, CASE WHEN is_win THEN 'W' WHEN is_loss THEN 'L' ELSE 'T' END AS result,
         ROW_NUMBER() OVER (PARTITION BY season, season_type, team_id ORDER BY game_datetime_utc, game_pk) AS team_game_number
  FROM {T('silver', 'fact_team_game')} tg
  WHERE is_completed
),
s AS (
  SELECT *, team_game_number
            - ROW_NUMBER() OVER (PARTITION BY season, season_type, team_id, result ORDER BY team_game_number) AS streak_group
  FROM g
)
SELECT
  season, season_type, game_type, team_id, game_pk, official_date, team_game_number,
  opponent_id, is_home, runs_scored, runs_allowed, run_differential, hits, errors, result,
  is_win, is_loss, is_extra_innings, is_one_run_game,
  SUM(INT(is_win))  OVER w_cum AS wins_to_date,
  SUM(INT(is_loss)) OVER w_cum AS losses_to_date,
  ROUND(SUM(INT(is_win)) OVER w_cum / team_game_number, 3) AS win_pct_to_date,
  SUM(run_differential) OVER w_cum AS run_differential_to_date,
  SUM(INT(is_win)) OVER w_last10 AS wins_last_10,
  COUNT(*) OVER w_last10 AS games_last_10,
  ROW_NUMBER() OVER (PARTITION BY season, season_type, team_id, result, streak_group ORDER BY team_game_number) AS streak_length,
  CONCAT(result, ROW_NUMBER() OVER (PARTITION BY season, season_type, team_id, result, streak_group ORDER BY team_game_number)) AS streak_code
FROM s
WINDOW w_cum    AS (PARTITION BY season, season_type, team_id ORDER BY team_game_number ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
       w_last10 AS (PARTITION BY season, season_type, team_id ORDER BY team_game_number ROWS BETWEEN 9 PRECEDING AND CURRENT ROW)
""")

with P.log_step(spark, RUN_ID, "gold", "team_game_log") as step:
    step.records = P.replace_table(spark, team_game_log, T("gold", "team_game_log"),
        "Gold: team game-by-game log with running record, run differential, rolling last-10 and streak")

## team_season_summary
Pythagorean expectation: `RS^1.83 / (RS^1.83 + RA^1.83)`. `luck_wins = wins − expected_wins`:
positive means the team won more games than its run differential suggests.

In [ ]:
team_season_summary = spark.sql(f"""
WITH log AS (SELECT * FROM {T('gold', 'team_game_log')}),
agg AS (
  SELECT season, season_type, team_id,
    COUNT(*) AS games, SUM(INT(is_win)) AS wins, SUM(INT(is_loss)) AS losses,
    SUM(runs_scored) AS runs_scored, SUM(runs_allowed) AS runs_allowed,
    SUM(INT(is_home AND is_win)) AS home_wins, SUM(INT(is_home AND is_loss)) AS home_losses,
    SUM(INT(NOT is_home AND is_win)) AS away_wins, SUM(INT(NOT is_home AND is_loss)) AS away_losses,
    SUM(INT(is_one_run_game AND is_win)) AS one_run_wins, SUM(INT(is_one_run_game AND is_loss)) AS one_run_losses,
    SUM(INT(is_extra_innings AND is_win)) AS extra_inning_wins, SUM(INT(is_extra_innings AND is_loss)) AS extra_inning_losses,
    SUM(INT(run_differential >= 5)) AS blowout_wins, SUM(INT(run_differential <= -5)) AS blowout_losses,
    MAX(CASE WHEN result = 'W' THEN streak_length END) AS longest_win_streak,
    MAX(CASE WHEN result = 'L' THEN streak_length END) AS longest_losing_streak,
    MAX_BY(streak_code, team_game_number) AS current_streak,
    MAX_BY(wins_last_10, team_game_number) AS last_10_wins,
    MAX_BY(games_last_10 - wins_last_10, team_game_number) AS last_10_losses,
    MAX(official_date) AS last_game_date
  FROM log GROUP BY season, season_type, team_id
),
att AS (
  SELECT tg.season, tg.season_type, tg.team_id, ROUND(AVG(d.attendance)) AS avg_home_attendance
  FROM {T('silver', 'fact_team_game')} tg JOIN {T('silver', 'game_details')} d USING (game_pk)
  WHERE tg.is_home AND tg.is_completed AND d.attendance > 0
  GROUP BY tg.season, tg.season_type, tg.team_id
)
SELECT a.season, a.season_type, a.team_id, t.team_name, t.team_abbreviation, t.league_abbreviation, t.division_name,
  a.games, a.wins, a.losses, ROUND(a.wins / a.games, 3) AS win_pct,
  a.runs_scored, a.runs_allowed, a.runs_scored - a.runs_allowed AS run_differential,
  ROUND(a.runs_scored / a.games, 2) AS runs_scored_per_game, ROUND(a.runs_allowed / a.games, 2) AS runs_allowed_per_game,
  ROUND(POW(a.runs_scored, {C.PYTHAGOREAN_EXPONENT}) /
        (POW(a.runs_scored, {C.PYTHAGOREAN_EXPONENT}) + POW(a.runs_allowed, {C.PYTHAGOREAN_EXPONENT})), 3) AS pythagorean_win_pct,
  ROUND(a.games * POW(a.runs_scored, {C.PYTHAGOREAN_EXPONENT}) /
        (POW(a.runs_scored, {C.PYTHAGOREAN_EXPONENT}) + POW(a.runs_allowed, {C.PYTHAGOREAN_EXPONENT})), 1) AS expected_wins,
  ROUND(a.wins - a.games * POW(a.runs_scored, {C.PYTHAGOREAN_EXPONENT}) /
        (POW(a.runs_scored, {C.PYTHAGOREAN_EXPONENT}) + POW(a.runs_allowed, {C.PYTHAGOREAN_EXPONENT})), 1) AS luck_wins,
  a.home_wins, a.home_losses, a.away_wins, a.away_losses,
  a.one_run_wins, a.one_run_losses, a.extra_inning_wins, a.extra_inning_losses, a.blowout_wins, a.blowout_losses,
  a.last_10_wins, a.last_10_losses, a.current_streak, a.longest_win_streak, a.longest_losing_streak,
  att.avg_home_attendance,
  s.division_rank, s.league_rank, s.games_back, s.wild_card_games_back, s.clinch_indicator,
  a.last_game_date
FROM agg a
LEFT JOIN {T('silver', 'dim_team')} t ON t.team_id = a.team_id
LEFT JOIN att ON att.season = a.season AND att.season_type = a.season_type AND att.team_id = a.team_id
LEFT JOIN {T('silver', 'official_standings')} s
       ON a.season_type = 'Regular Season' AND s.season = a.season AND s.team_id = a.team_id
""")

with P.log_step(spark, RUN_ID, "gold", "team_season_summary") as step:
    step.records = P.replace_table(spark, team_season_summary, T("gold", "team_season_summary"),
        "Gold: team record and splits per season and season type, run differential, Pythagorean expected wins and luck")

## head_to_head

In [ ]:
head_to_head = spark.sql(f"""
SELECT tg.season, tg.season_type, tg.team_id, t.team_name, tg.opponent_id, o.team_name AS opponent_name,
  COUNT(*) AS games, SUM(INT(is_win)) AS wins, SUM(INT(is_loss)) AS losses,
  ROUND(SUM(INT(is_win)) / COUNT(*), 3) AS win_pct,
  SUM(runs_scored) AS runs_scored, SUM(runs_allowed) AS runs_allowed, SUM(run_differential) AS run_differential,
  SUM(INT(is_home)) AS home_games, MIN(official_date) AS first_meeting, MAX(official_date) AS last_meeting,
  MAX_BY(CONCAT(CASE WHEN is_win THEN 'W ' ELSE 'L ' END, runs_scored, '-', runs_allowed), game_datetime_utc) AS last_result
FROM {T('silver', 'fact_team_game')} tg
LEFT JOIN {T('silver', 'dim_team')} t ON t.team_id = tg.team_id
LEFT JOIN {T('silver', 'dim_team')} o ON o.team_id = tg.opponent_id
WHERE tg.is_completed
GROUP BY tg.season, tg.season_type, tg.team_id, t.team_name, tg.opponent_id, o.team_name
""")

with P.log_step(spark, RUN_ID, "gold", "head_to_head") as step:
    step.records = P.replace_table(spark, head_to_head, T("gold", "head_to_head"),
        "Gold: team vs opponent record per season and season type (games, W-L, runs, first/last meeting, last result)")

In [ ]:
display(spark.sql(f"""
    SELECT season, team_abbreviation, wins, losses, run_differential, expected_wins, luck_wins, current_streak
    FROM {T('gold', 'team_season_summary')} WHERE season_type = 'Regular Season'
    ORDER BY season DESC, wins DESC LIMIT 15"""))